## Interval Analysis

In [2]:
# !pip install tensorboardX
# !pip install bound-propagation

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = False
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
bound_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Sequential):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200,10)

    def forward(self, input):
        input = (input - 0.1307)/0.3081
        input = F.relu(self.fc(input))
        input = self.fc2(input)
        input = F.softmax(input, dim=-1) # added softmax for probabilities
        return input

# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = Net()

model = model.to(device)
model.train()


Net(
  (fc): Linear(in_features=784, out_features=200, bias=True)
  (fc2): Linear(in_features=200, out_features=10, bias=True)
)

In [3]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')
    

In [4]:
train_model(model, 15)

Epoch 1/15, Loss: 2.012
Epoch 2/15, Loss: 1.729
Epoch 3/15, Loss: 1.627
Epoch 4/15, Loss: 1.596
Epoch 5/15, Loss: 1.582
Epoch 6/15, Loss: 1.572
Epoch 7/15, Loss: 1.566
Epoch 8/15, Loss: 1.561
Epoch 9/15, Loss: 1.557
Epoch 10/15, Loss: 1.553
Epoch 11/15, Loss: 1.550
Epoch 12/15, Loss: 1.547
Epoch 13/15, Loss: 1.545
Epoch 14/15, Loss: 1.542
Epoch 15/15, Loss: 1.540


In [5]:
test_model(model)

Accuracy on images: 93.43


### Write the interval analysis for the simple model

In [6]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

from bound_propagation import BoundModelFactory, HyperRectangle

factory = BoundModelFactory()
isinstance(model, nn.Sequential)
net = factory.build(model)

In [38]:
net.eval()
epsilons = np.linspace(0.01, 0.1, 10)
always_correct = np.zeros(len(epsilons), dtype=int)
still_correct = np.zeros(len(epsilons), dtype=int)
correct = 0
total = 0

for images, labels in test_loader:
    images = images.view((-1, 28 * 28)).to(device)
    labels = labels.to(device)

    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    predicted_correct = predicted == labels

    total += labels.size(0)
    correct += predicted_correct.sum().item()

    for idx, epsilon in enumerate(epsilons):
        input_bounds = HyperRectangle.from_eps(images, float(epsilon))
        crown_ibp_bounds = net.crown_ibp(input_bounds).concretize()
        lower, upper = crown_ibp_bounds.lower, crown_ibp_bounds.upper

        label_lower = lower.gather(1, labels.unsqueeze(1)).squeeze(1)
        label_mask = torch.zeros_like(upper, dtype=torch.bool)
        label_mask.scatter_(1, labels.unsqueeze(1), True)
        max_other_upper = upper.masked_fill(label_mask, float('-inf')).max(dim=1).values
        robust_mask = label_lower > max_other_upper
        always_correct[idx] += robust_mask.sum().item()

print(f'Standard accuracy: {correct / total*100:.2f}%')
for eps, ac in zip(epsilons, always_correct):
    print(f'eps={eps:.3f}: always_correct={ac / total * 100:.2f}%')

        
        
        
        
    


Standard accuracy: 93.43%
eps=0.010: always_correct=74.60%
eps=0.020: always_correct=57.03%
eps=0.030: always_correct=39.42%
eps=0.040: always_correct=25.46%
eps=0.050: always_correct=14.88%
eps=0.060: always_correct=7.51%
eps=0.070: always_correct=3.55%
eps=0.080: always_correct=1.69%
eps=0.090: always_correct=0.61%
eps=0.100: always_correct=0.20%
